# Global and convex optimization

[Open in Colab](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/global-and-convex-optimization.ipynb) · [Open in Binder](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/global-and-convex-optimization.ipynb)

By Joaquim Gromicho. Modernized from the original teaching notebook.

Explore local minima, recognize convexity, reformulate the cylinder and a constrained quadratic, then actually compare Ipopt, CPLEX, Xpress and Gurobi.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'pyomo': 'pyomo', 'sympy': 'sympy', 'numpy': 'numpy', 'matplotlib': 'matplotlib', 'pandas': 'pandas'}
ensure_packages(required_packages)


In [ ]:
import pyomo.environ as pyo
import math
import sympy
import numpy
import matplotlib.pyplot as plt
from teaching_utils import install_coin_solvers, solve_checked
install_coin_solvers()


In [ ]:
def Minimize(function,start=0,precision=6):
    model=pyo.ConcreteModel('Local polynomial optimization')
    model.x=pyo.Var(initialize=start)
    model.obj=pyo.Objective(expr=function(model.x))
    result=solve_checked(model,'ipopt')
    print(result.solver.termination_condition)
    print(f'f({pyo.value(model.x):.{precision}f}) = {pyo.value(model.obj):.{precision}f}')
    return pyo.value(model.x),pyo.value(model.obj)


In [ ]:
def polynomial(x):
    return x**4+3*x**3-15*x**2-19*x+130
x=sympy.Symbol('x')
f=polynomial(x)
description=sympy.latex(f)


In [ ]:
sympy.diff( sympy.diff( f ) )

In [ ]:
sympy.solveset(sympy.diff(f,x,2),x,domain=sympy.S.Reals)


In [ ]:
local_6=Minimize(polynomial,start=0,precision=10)


A solver reports a local stationary solution; compare other starting points.


In [ ]:
local_8=Minimize(polynomial,start=10,precision=10)


Does this starting point lead to the same basin of attraction?


In [ ]:
local_10=Minimize(polynomial,start=-10,precision=10)


Different local minima can have different values. The plot and stationary-point calculation below provide additional information.


In [ ]:
def Plot(function,start=-5,stop=5,num=300):
    grid=numpy.linspace(start,stop,num)
    plt.plot(grid,function(grid),label='$'+description+'$',linewidth=2)
    plt.legend();plt.xlabel('x');plt.ylabel('objective');plt.show()


In [ ]:
Plot(polynomial)
stationary=[float(sympy.re(z)) for z in sympy.nroots(sympy.diff(f,x)) if abs(float(sympy.im(z)))<1e-10]
global_candidate=min(stationary,key=polynomial)
print('Stationary points:',stationary,'global minimizer:',global_candidate)
assert max(v for _,v in [local_6,local_8,local_10])-min(v for _,v in [local_6,local_8,local_10])>1
# Positive leading quartic coefficient gives f(x) -> infinity at both ends.


Using a function instead of unrestricted text evaluation makes the mathematical expression reusable across NumPy, SymPy, Pyomo and CVXPY.


# The importance of convexity

Local optimality need not be global. For a convex minimization problem every local minimum is global; the structure also supports efficient algorithms under suitable assumptions. General global optimization includes computationally hard problem families, but calling all global optimization “NP-complete” is not a valid classification.

Now introduce CVXPY and its [disciplined convex programming rules](https://www.cvxpy.org/tutorial/dcp/index.html). These are sufficient recognition rules: rejection does not by itself prove that the feasible set is nonconvex. Equivalent expressions can be recognized differently. The polynomial above is deliberately not a DCP minimization objective.


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'cvxpy': 'cvxpy'}
ensure_packages(required_packages)


In [ ]:
import cvxpy as cp


In [ ]:
x_cp=cp.Variable()
prob=cp.Problem(cp.Minimize(polynomial(x_cp)))
assert not prob.is_dcp()
try:
    prob.solve(solver='CLARABEL')
except cp.error.DCPError as error:
    print(type(error).__name__,': expected rejection of a non-DCP objective')


There are many `python` resources for __convex optimization__ being developed at the University of Stanford by Professor Boyd and his team.

We will use two: [`cvxopt`](https://cvxopt.org/userguide/index.html#) and [`cvxpy`](https://www.cvxpy.org/tutorial/dcp/index.html).

The latter seems very powerful not only as a solver but also as a modeling tool. We use it on this notebook. 
The first, more well-known, will be used on the second part. 

Let us consider Betty's problem to find the cylinder with maximum volume:

$$
\begin{array}{rl}
\max    &   \pi r^2 h \\
s.t.    & 2\pi r^2 + 2\pi r h \leq 12 \\
        & r \geq 0 \\
        & h \geq 0
\end{array}
$$


In [ ]:
r,h=cp.Variable(nonneg=True),cp.Variable(nonneg=True)
prob=cp.Problem(cp.Maximize(math.pi*r**2*h),[2*math.pi*r**2+2*math.pi*r*h<=12])
print('Direct cylinder expression is DCP:',prob.is_dcp())


This package raises an exception when trying to solve a problem that is found not to be  (disciplined) convex.

Therefore is better to surround the call to `solve` by a `try` - `except` block.  

In [ ]:
try:
    prob.solve(solver='CLARABEL')
except cp.error.DCPError:
    print('Expected DCP rejection: reformulate the cylinder first.')


In [ ]:
if prob.is_dcp():
    print( prob.solve() )
    print( r.value, h.value )
else:
    print( prob )
    print( 'is not Disciplined Convex Programming')

As we saw on the lecture the reformulation in a singel variable problem is convex, `cvxpy` also confirms that.

In [ ]:
r=cp.Variable(nonneg=True)
prob=cp.Problem(cp.Maximize(6*r-math.pi*cp.power(r,3)),[r<=math.sqrt(6/math.pi)])
assert prob.is_dcp()
value=prob.solve(solver='CLARABEL')
assert prob.status==cp.OPTIMAL
h=(6-math.pi*r.value**2)/(math.pi*r.value)
print(f'r={r.value:.6f}, h={h:.6f}, volume={value:.6f}')
assert math.isclose(r.value,math.sqrt(2/math.pi),rel_tol=1e-4)


We saw a second problem that became convex after reformulating, namely:

$$
\begin{array}{rl}
\min    & x_1^2 + x_2^2 \\
s.t.    & x_1/(1+x_2^2) \leq 0 \\
        & (x_1+x_2)^2 = 0 
\end{array}
$$

which is clearly equivalent to:
$$
\begin{array}{rrcrcr}
\min    & x_1^2 & + & x_2^2 \\
s.t.    & x_1   &   &       & \leq & 0 \\
        & x_1   & + & x_2   &    = & 0 
\end{array}
$$

Also in this case `cvxpy` does the right thing.

In [ ]:
x_cp=cp.Variable(2)
prob=cp.Problem(cp.Minimize(cp.sum_squares(x_cp)),[x_cp[0]/(1+x_cp[1]**2)<=0,(x_cp[0]+x_cp[1])**2==0])
assert not prob.is_dcp()
try:
    prob.solve(solver='CLARABEL')
except cp.error.DCPError:
    print('Expected DCP rejection of this representation.')


In [ ]:
x_cp=cp.Variable(2)
prob=cp.Problem(cp.Minimize(cp.sum_squares(x_cp)),[x_cp[0]<=0,x_cp[0]+x_cp[1]==0])
assert prob.is_dcp()
prob.solve(solver='CLARABEL')
assert prob.status==cp.OPTIMAL and abs(prob.value)<1e-6
print(x_cp.value,prob.value)
# The denominator is positive, so x1/(1+x2**2)<=0 means x1<=0, not x1>=0.


In [ ]:
comparison=[]
def SolveWith(model,solver_name):
    from time import perf_counter
    instance=model.clone()
    started=perf_counter()
    result=solve_checked(instance,solver_name)
    value=pyo.value(instance.obj)
    comparison.append({'solver':solver_name,'objective':value,'seconds':perf_counter()-started})
    print(solver_name,result.solver.termination_condition,[pyo.value(instance.x[j]) for j in instance.J],value)
    return instance


In [ ]:
m     = pyo.ConcreteModel('test')
m.J   = [1,2]
m.x   = pyo.Var(m.J)
m.obj = pyo.Objective( expr = sum( m.x[j]**2 for j in m.J ), sense=pyo.minimize )
m.c1  = pyo.Constraint( expr = m.x[1] <= 0 )
m.c2  = pyo.Constraint( expr = sum( m.x[j] for j in m.J ) == 0 )

In [ ]:
SolveWith(m,'ipopt')

## Introduce commercial quadratic solvers now
The original lesson adds these interfaces only after the Ipopt comparison. The tiny model fits their packaged limited editions. Every engine below must actually solve a fresh model; an unavailable or unlicensed engine is an error, not a successful comparison.


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'cplex': 'cplex', 'gurobipy': 'gurobipy', 'xpress': 'xpress'}
ensure_packages(required_packages)


In [ ]:
from teaching_utils import available_pyomo_solvers
print(available_pyomo_solvers(['ipopt','cplex_direct','gurobi_direct','xpress_direct']))


In [ ]:
SolveWith(m,'cplex_direct')

In [ ]:
SolveWith(m,'xpress_direct')

In [ ]:
SolveWith(m,'gurobi_direct')

In [ ]:
import pandas as pd
display(pd.DataFrame(comparison))
assert {row['solver'] for row in comparison}=={'ipopt','cplex_direct','xpress_direct','gurobi_direct'}
assert all(abs(row['objective'])<1e-5 for row in comparison)
